#Introduction to GPU Acceleration
### 🔍 Why Use GPUs?

GPUs are optimized for large-scale parallel computation, making them ideal for matrix-heavy tasks in deep learning. In this lab, you'll compare training times and performance on CPU vs GPU and learn how to write GPU-efficient code.


##Check device availability

## TensorFlow

In [1]:
import tensorflow as tf
print("Is GPU available?", tf.config.list_physical_devices('GPU'))

Is GPU available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


##PyTorch

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 🚀 Moving Models and Data to GPU

To fully utilize the GPU, both the model and input data must be moved to the GPU device. This ensures the computation is performed on the GPU instead of the CPU.

Let's see how to do this in both TensorFlow and PyTorch.


##TensorFlow - Using GPU Automatically

In [3]:
# TensorFlow uses GPU by default when available
import tensorflow as tf

with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)


Operation completed on: /job:localhost/replica:0/task:0/device:GPU:0


##PyTorch - Manual GPU Transfer

In [4]:
import torch

# Use 'cuda' if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example tensor operation on GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)
c = torch.matmul(a, b)

print("Tensor 'c' is on device:", c.device)


Tensor 'c' is on device: cuda:0


##Measuring Training Time on CPU vs GPU

## ⏱️ Performance Benchmark: CPU vs GPU

We'll train a simple model on the MNIST dataset using both CPU and GPU. This will help us visualize the speedup provided by GPU acceleration.

Steps:
- Train on CPU and measure the time
- Train on GPU and measure the time
- Compare the difference


In [2]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)




100%|██████████| 9.91M/9.91M [00:00<00:00, 19.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 505kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.42MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.0MB/s]


##Train on CPU

In [3]:
# Train on CPU
def train_on_cpu():
    device = torch.device("cpu")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

✅ CPU training time: 11.71 sec


##Train on GPU
Change your runtine to T4 GPU and run the following code block

In [3]:
# Train on GPU
def train_on_gpu():
    if not torch.cuda.is_available():
        print("🚫 CUDA not available on this system.")
        return

    device = torch.device("cuda")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()


✅ GPU training time: 7.41 sec


I suppose we got lesser time for the gpu, it makes more difference on larger models that have more number of layers and filters, gpu speeds up the matrix multiplication due to the presence of numerous small cores.

**So here's an activity for you**
##Use tensorflow to train a model on the MNIST digits dataset on both gpu and cpu and examine which one works faster.

Use your custom number of layers and filters to experiment with the hyperparameters of the model.

In [2]:
import time
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load and preprocess MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Reshape to include channel dimension and normalize pixels to [0, 1]
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# 2. Define a function to build a deeper CNN model
def build_custom_cnn():
    model = models.Sequential([
        # Layer 1: 64 Filters
        layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        # Layer 2: 128 Filters
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Layer 3: 128 Filters
        layers.Conv2D(128, (3, 3), activation='relu'),
        # Layer 4: 256 Filters
        layers.Conv2D(256, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Dense Layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Data loaded & model builder defined.")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Data loaded & model builder defined.


In [3]:
# Force TensorFlow to run on CPU
print(" Starting CPU Training...")
with tf.device('/CPU:0'):
    cpu_model = build_custom_cnn()

    start_time = time.time()
    cpu_model.fit(x_train, y_train, epochs=3, batch_size=128, verbose=1)
    cpu_time = time.time() - start_time

print(f"✅ CPU Training Time: {cpu_time:.2f} seconds")

 Starting CPU Training...
Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 669s 1s/step - accuracy: 0.9620 - loss: 0.1216
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 659s 1s/step - accuracy: 0.9893 - loss: 0.0341
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 681s 1s/step - accuracy: 0.9934 - loss: 0.0216
✅ CPU Training Time: 2008.70 seconds


In [4]:
# Force TensorFlow to run on GPU (Ensure Runtime is set to T4 GPU in Colab)
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(" Starting GPU Training...")
    with tf.device('/GPU:0'):
        gpu_model = build_custom_cnn()

        start_time = time.time()
        gpu_model.fit(x_train, y_train, epochs=3, batch_size=128, verbose=1)
        gpu_time = time.time() - start_time

    print(f" GPU Training Time: {gpu_time:.2f} seconds\n")

    # Speedup ratio calculation
    speedup = 2008 / gpu_time
    print(f" Summary: The GPU was {speedup:.2f}x faster than the CPU!")
else:
    print(" No GPU detected. Make sure to go to Runtime > Change runtime type > T4 GPU.")

 Starting GPU Training...
Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.9625 - loss: 0.1194
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.9896 - loss: 0.0342
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - accuracy: 0.9936 - loss: 0.0212
 GPU Training Time: 31.58 seconds

 Summary: The GPU was 63.58x faster than the CPU!
